# Logit Básico Cloud — Entrenamiento en Kaggle Notebooks

**Corre en Kaggle Notebooks** (CPU).

**Dataset requerido:** `mecmt07-features` (subir `features_train.parquet` y `features_test.parquet`).

**Flujo:**
1. Preprocesamiento: SimpleImputer (mediana) + StandardScaler
2. Logit sin regularización efectiva (C=1e6): 5-fold CV AUC
3. Refit final en dataset completo
4. Guardar pipeline + metadata → descargar desde Output tab

In [ ]:
import numpy as np
import pandas as pd
import json
import joblib
import time
import warnings
warnings.filterwarnings('ignore')
from pathlib import Path

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from scipy.stats import gaussian_kde

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import roc_auc_score, roc_curve, confusion_matrix
import sklearn

print(f'sklearn version : {sklearn.__version__}')
print('Imports OK')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CONFIGURACIÓN
# ══════════════════════════════════════════════════════════════════════════════
SEED    = 42
N_FOLDS = 5
C_REG   = 1e6   # sin regularización efectiva
SOLVER  = 'lbfgs'

DATA_DIR  = Path('/kaggle/input/datasets/davidguzzi/mecmt07-features')
MODEL_DIR = Path('/kaggle/working')

# Matplotlib
plt.rcParams['figure.dpi'] = 100
plt.rcParams['axes.spines.top']   = False
plt.rcParams['axes.spines.right'] = False

np.random.seed(SEED)

print('=' * 65)
print('CONFIGURACIÓN')
print('=' * 65)
print(f'  DATA_DIR  : {DATA_DIR}')
print(f'  MODEL_DIR : {MODEL_DIR}')
print(f'  N_FOLDS   : {N_FOLDS}')
print(f'  SEED      : {SEED}')
print(f'  C         : {C_REG}  (sin regularizacion efectiva)')
print(f'  solver    : {SOLVER}')
print(f'  CPU       : Logit usa CPU (sklearn lbfgs)')
print('=' * 65)
for f in ['features_train.parquet', 'features_test.parquet']:
    exists = (DATA_DIR / f).exists()
    print(f'  {f}: {"OK" if exists else "NO encontrado"}')

In [ ]:
print('Cargando datos...')
df      = pd.read_parquet(DATA_DIR / 'features_train.parquet')
df_test = pd.read_parquet(DATA_DIR / 'features_test.parquet')

feature_cols = [c for c in df.columns if c not in ('SK_ID_CURR', 'TARGET')]

# Encodear columnas categóricas
cat_cols = [c for c in feature_cols if df[c].dtype == 'object']
if cat_cols:
    print(f'  Columnas categóricas encontradas: {cat_cols}')
    for col in cat_cols:
        cats    = pd.concat([df[col], df_test[col]]).dropna().unique()
        mapping = {v: i for i, v in enumerate(cats)}
        df[col]      = df[col].map(mapping)
        df_test[col] = df_test[col].map(mapping)
    print('  Encoding completado ✓')
else:
    print('  Sin columnas categóricas')

X       = df[feature_cols].values
y       = df['TARGET'].values
X_test  = df_test[feature_cols].values
sk_ids_test = df_test['SK_ID_CURR'].values

print(f'\nTrain: {X.shape}  | Default rate: {y.mean():.2%}')
print(f'Test : {X_test.shape}')

In [ ]:
def compute_metrics(y_true, y_prob, threshold=0.5, label='Model'):
    y_pred = (y_prob >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    auc  = roc_auc_score(y_true, y_prob)
    rec  = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    prec = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    f1   = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0.0
    return dict(Model=label, AUC=round(auc, 4),
                N=len(y_true), P=int(y_pred.sum()),
                TP=int(tp), TN=int(tn), FP=int(fp), FN=int(fn),
                Recall=round(rec, 4), Precision=round(prec, 4), F1=round(f1, 4))

print('Funciones auxiliares definidas.')

## 1. Validación Cruzada — 5-Fold

In [ ]:
pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler()),
    ('lr',      LogisticRegression(C=C_REG, max_iter=2000, solver=SOLVER,
                                   class_weight='balanced', random_state=SEED))
])

print('=' * 65)
print('VALIDACIÓN CRUZADA — 5-FOLD (Logit Básico)')
print('=' * 65)
print(f'  N_FOLDS : {N_FOLDS}')
print(f'  C       : {C_REG}')
print(f'  solver  : {SOLVER}')
print(f'  Datos   : {X.shape[0]:,} filas')
print('─' * 65)

skf      = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
oof_prob = np.zeros(len(y))
cv_aucs  = []
t0       = time.time()

for fold, (tr_idx, val_idx) in enumerate(skf.split(X, y)):
    pipe_fold = Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler',  StandardScaler()),
        ('lr',      LogisticRegression(C=C_REG, max_iter=2000, solver=SOLVER,
                                       class_weight='balanced', random_state=SEED))
    ])
    pipe_fold.fit(X[tr_idx], y[tr_idx])
    prob = pipe_fold.predict_proba(X[val_idx])[:, 1]
    oof_prob[val_idx] = prob
    auc  = roc_auc_score(y[val_idx], prob)
    cv_aucs.append(auc)
    print(f'  Fold {fold+1}/{N_FOLDS} │ AUC={auc:.5f}')

cv_mean    = np.mean(cv_aucs)
cv_std     = np.std(cv_aucs)
cv_auc_oof = roc_auc_score(y, oof_prob)
elapsed_cv = time.time() - t0

print('─' * 65)
print(f'  CV AUC mean    : {cv_mean:.5f} ± {cv_std:.5f}')
print(f'  CV AUC OOF     : {cv_auc_oof:.5f}')
print(f'  Tiempo         : {elapsed_cv:.0f}s')

## 2. Modelo Final — Refit en Train Completo

In [ ]:
# ── Refit en dataset completo ──────────────────────────────────────────────────
print('Refitteando en dataset completo...')
t0 = time.time()
pipeline.fit(X, y)
elapsed_fit = time.time() - t0
print(f'  Refit completado ✓  ({elapsed_fit:.0f}s)')

y_prob_train = pipeline.predict_proba(X)[:, 1]
train_auc    = roc_auc_score(y, y_prob_train)
print(f'  Train AUC: {train_auc:.5f}')

## 3. Metricas sobre Train Completo

In [ ]:
metrics_train = compute_metrics(y, y_prob_train, label='Logit (train in-sample)')
metrics_oof   = compute_metrics(y, oof_prob,     label='Logit (OOF 5-fold)')

for m in [metrics_train, metrics_oof]:
    print(f"\n{m['Model']}")
    print(f"  AUC={m['AUC']:.4f}  Recall={m['Recall']:.4f}  "
          f"Precision={m['Precision']:.4f}  F1={m['F1']:.4f}")
    print(f"  TP={m['TP']:,}  TN={m['TN']:,}  FP={m['FP']:,}  FN={m['FN']:,}")

display(pd.DataFrame([metrics_train, metrics_oof]).set_index('Model'))

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
for y_prob, label, color in [
        (y_prob_train, f'Train (AUC={train_auc:.4f})',    '#e74c3c'),
        (oof_prob,     f'OOF CV (AUC={cv_auc_oof:.4f})', '#3498db')]:
    fpr, tpr, _ = roc_curve(y, y_prob)
    ax.plot(fpr, tpr, color=color, lw=2, label=label)
ax.plot([0, 1], [0, 1], 'k--', lw=1)
ax.set_xlabel('FPR (1 - Especificidad)')
ax.set_ylabel('TPR (Sensibilidad)')
ax.set_title('ROC Curve — Logit Basico')
ax.legend(loc='lower right')
plt.tight_layout()
plt.savefig(MODEL_DIR / 'logit_roc_curve.png', dpi=120)
plt.show()
print('Grafico guardado: logit_roc_curve.png')

## 4. Coeficientes del Modelo

In [ ]:
lr_coef = pipeline.named_steps['lr'].coef_[0]
coef_s  = pd.Series(lr_coef, index=feature_cols).sort_values(key=abs, ascending=False)
top20   = coef_s.head(20).sort_values()

fig, ax = plt.subplots(figsize=(9, 7))
colors  = ['#e74c3c' if v > 0 else '#3498db' for v in top20.values]
ax.barh(top20.index, top20.values, color=colors)
ax.axvline(0, color='black', lw=0.8)
ax.set_xlabel('Coeficiente (escala estandarizada)')
ax.set_title('Top-20 Coeficientes — Logit Basico\n(rojo=positivo, azul=negativo)')
plt.tight_layout()
plt.savefig(MODEL_DIR / 'logit_coeficientes.png', dpi=120)
plt.show()
print('Grafico guardado: logit_coeficientes.png')

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
for val, color, label in [(0, '#3498db', 'TARGET=0 (paga)'),
                           (1, '#e74c3c', 'TARGET=1 (default)')]:
    probs = y_prob_train[y == val]
    kde   = gaussian_kde(probs, bw_method=0.1)
    xs    = np.linspace(0, 1, 300)
    ax.plot(xs, kde(xs), color=color, lw=2, label=label)
    ax.fill_between(xs, kde(xs), alpha=0.15, color=color)
ax.set_xlabel('Score predicho P(default)')
ax.set_ylabel('Densidad')
ax.set_title('Distribucion de scores — Logit Basico')
ax.legend()
plt.tight_layout()
plt.savefig(MODEL_DIR / 'logit_score_distribution.png', dpi=120)
plt.show()
print('Grafico guardado: logit_score_distribution.png')

## 5. Guardar Modelo y Metadata

In [ ]:
model_path = MODEL_DIR / 'logit_cloud_best.pkl'
meta_path  = MODEL_DIR / 'logit_cloud_metadata.json'

joblib.dump({'pipeline': pipeline, 'feature_cols': feature_cols}, model_path)

metadata = {
    'model_type'     : 'logit_basic',
    'cv_auc_mean'    : round(cv_mean, 6),
    'cv_auc_std'     : round(cv_std, 6),
    'cv_auc_oof'     : round(cv_auc_oof, 6),
    'train_auc'      : round(train_auc, 6),
    'C'              : C_REG,
    'solver'         : SOLVER,
    'n_folds'        : N_FOLDS,
    'feature_cols'   : feature_cols,
    'sklearn_version': sklearn.__version__,
    'timestamp'      : pd.Timestamp.now().isoformat()
}
with open(meta_path, 'w') as f:
    json.dump(metadata, f, indent=2)

print('=' * 65)
print('ARTEFACTOS GUARDADOS')
print('=' * 65)
print(f'  {model_path.name:<45} ({model_path.stat().st_size/1e6:.2f} MB)')
print(f'  {meta_path.name}')
print('=' * 65)
print('\n>>> Descargar desde el Output tab de Kaggle:')
print(f'    - {model_path.name}')
print(f'    - {meta_path.name}')
print('\n>>> Luego correr localmente: logit_cloud_predict.ipynb')

## Resumen Final — Logit Básico Cloud

In [ ]:
import platform
print('=' * 65)
print('LOGIT BÁSICO CLOUD — RESUMEN FINAL')
print('=' * 65)
print(f'  CV AUC (5-fold OOF) : {cv_auc_oof:.5f}')
print(f'  CV AUC mean         : {cv_mean:.5f} ± {cv_std:.5f}')
print(f'  Train AUC (full)    : {train_auc:.5f}')
print(f'  Gap                 : {abs(train_auc - cv_auc_oof):.5f}')
print(f'  n_train             : {X.shape[0]:,}')
print('─' * 65)
print(f'  C      : {C_REG}')
print(f'  solver : {SOLVER}')
print('─' * 65)
print(f'  sklearn   : {sklearn.__version__}')
print(f'  numpy     : {np.__version__}')
print(f'  Python    : {platform.python_version()}')
print(f'  Timestamp : {pd.Timestamp.now().strftime("%Y-%m-%d %H:%M:%S")}')
print('=' * 65)